In [6]:
import pandas as pd
import numpy as np
import re

# ==========================
# LOAD
# ==========================

file="../4.Near Miss Database - 2013.xlsx"
sheet="Near Miss"

df=pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

print("Original Shape:",df.shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

df=df.replace(
    [
        "NA",
        "N/A",
        "na",
        "n/a",
        "nan",
        "NULL",
        "null",
        "Not Mentioned",
        "not mentioned",
        "NOT MENTIONED"
    ],
    np.nan
)

# ==========================
# REMOVE EMPTY COLUMNS
# ==========================

remove=[]

for col in df.columns:

    if (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()
    ):
        remove.append(col)

df=df.drop(
    columns=remove,
    errors="ignore"
)

print("\nRemoved Columns:")
print(remove)

# ==========================
# IDENTIFY COLUMNS
# ==========================

si_col=None
desc_col=None

for col in df.columns:

    clean=(
        col.lower()
        .replace("_","")
        .replace(":","")
    )

    if clean in [
        "slno",
        "sino",
        "serialno"
    ]:
        si_col=col

    if "description" in clean:
        desc_col=col

# ==========================
# REMOVE EMPTY DESCRIPTION
# ==========================

if desc_col:

    before=len(df)

    df=df[
        df[desc_col]
        .fillna("")
        .str.strip()
        !=""
    ]

    print(
        "\nRows Removed:",
        before-len(df)
    )

# ==========================
# MERGE POSSIBLE CONSEQUENCE
# ==========================

pc=[]

for col in df.columns:

    if (
        "possible_consequence"
        in col.lower()
    ):
        pc.append(col)

if len(pc)>1:

    merged=(
        df[pc]
        .fillna("")
        .apply(
            lambda r:
            " | ".join(
                [
                    str(v).strip()

                    for v in r

                    if (
                        str(v).strip()
                        and
                        str(v).lower()
                        not in [
                            "nan",
                            "n/a"
                        ]
                    )
                ]
            ),
            axis=1
        )
    )

    df[pc[0]]=merged

    df=df.drop(
        columns=pc[1:]
    )

print("\nPossible Consequence Merged")

# ==========================
# DATE CLEANING
# ==========================

date_cols=[]

required_dates=[

    "date_of_the_near_miss",
    "date_of_report",
    "target_date",
    "date_of_investigation_completion",
    "date_corrective_action_completed",
    "date_investigation_commenced",
    "extension_(if_any)_with_remarks"
]

for col in df.columns:

    if col.lower() in required_dates:

        date_cols.append(col)

def fix_date(v):

    if pd.isna(v):
        return np.nan

    v=str(v).strip()

    if v.lower() in [
        "na",
        "n/a",
        "not mentioned",
        "nan"
    ]:
        return np.nan

    # already correct → keep
    if re.match(
        r"^\d{2}-[A-Za-z]{3}-\d{2}$",
        v
    ):
        return v

    try:

        d=pd.to_datetime(
            v,
            errors="coerce"
        )

        if pd.notna(d):

            return d.strftime(
                "%d-%b-%y"
            )

    except:
        pass

    return np.nan

for col in date_cols:

    df[col]=(
        df[col]
        .apply(fix_date)
    )

print("\nFormatted Dates:")
print(date_cols)

# ==========================
# DUPLICATES
# ==========================

dup=df.duplicated().sum()

df=df.drop_duplicates()

print("\nDuplicates:",dup)

# ==========================
# RESET SI NO
# ==========================

if si_col:

    df=df.reset_index(
        drop=True
    )

    df[si_col]=range(
        1,
        len(df)+1
    )

# ==========================
# FINAL EMPTY FORMAT
# ==========================

df=df.replace(
    np.nan,
    ""
)

# ==========================
# REPORT
# ==========================

missing=(
    df.replace(
        "",
        np.nan
    )
    .isna()
    .sum()
)

missing=missing[
    missing>0
]

print("\nMissing Summary:")
print(missing)

print("\nFinal Shape:")
print(df.shape)

print("\nFinal Columns:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================

out="cleaned_4_Near_Miss_2013.xlsx"

df.to_excel(
    out,
    index=False
)

print("\nSaved:",out)

Original Shape: (781, 31)

Removed Columns:
[]

Rows Removed: 11

Possible Consequence Merged

Formatted Dates:
['Date_of_the_Near_Miss', 'Date_of_Report', 'Date_Corrective_action_completed', 'Date_Investigation_Commenced', 'Target_Date', 'Date_of_Investigation_Completion', 'Extension_(if_any)_with_remarks']

Duplicates: 0

Missing Summary:
Date_of_the_Near_Miss                       3
Date_of_Report                              2
Delay_(in_days)                            23
Incident_Category_(Potential)              87
Location_of_the_Unsafe_act_/_condition     14
Possible_Consequence                       29
Probability_of_Reoccurence                105
Cause_Analysis                              3
Counter_measure_to_prevent_recurrence       5
Date_Corrective_action_completed          287
Area_of_Concern                           350
Date_Investigation_Commenced              175
Date_of_Investigation_Completion           50
Extension_(if_any)_with_remarks           768
Time_Period_(